In [9]:
# ============================================
# Evaluation: PSNR / SSIM / LPIPS
# ============================================

import imageio.v2 as imageio
import numpy as np
import torch
import lpips
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


# ---------- 配置 ----------
IMG_A_PATH = "fcgb_pro.png"       # 渲染结果 A
IMG_B_PATH = "single_base.png"    # 渲染结果 B（可选，不要就删掉相关代码）
GT_PATH    = "gt.png"             # Ground Truth
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
LPIPS_NET  = "alex"               # alex / vgg / squeeze
# -------------------------


# ---------- numpy -> torch (LPIPS 专用) ----------
def np_to_lpips_tensor(img_np, device):
    """
    img_np: numpy.ndarray, [H,W,3], float32, range [0,1]
    return: torch.Tensor, [1,3,H,W], range [-1,1]
    """
    t = torch.from_numpy(img_np)       # [H,W,3]
    t = t.permute(2, 0, 1)              # [3,H,W]
    t = t.unsqueeze(0)                  # [1,3,H,W]
    t = t * 2.0 - 1.0                   # [-1,1]
    return t.to(device)


# ---------- 读取图像 ----------
def load_image_np(path):
    img = imageio.imread(path)
    img = img.astype(np.float32) / 255.0   # [0,1]
    assert img.ndim == 3 and img.shape[2] == 3, f"{path} must be RGB"
    return img


img_a = load_image_np(IMG_A_PATH)
img_b = load_image_np(IMG_B_PATH)
gt    = load_image_np(GT_PATH)


# ---------- PSNR / SSIM ----------
psnr_a = peak_signal_noise_ratio(gt, img_a, data_range=1.0)
psnr_b = peak_signal_noise_ratio(gt, img_b, data_range=1.0)

ssim_a = structural_similarity(gt, img_a, data_range=1.0, channel_axis=-1)
ssim_b = structural_similarity(gt, img_b, data_range=1.0, channel_axis=-1)


# ---------- LPIPS ----------
lpips_fn = lpips.LPIPS(net=LPIPS_NET).to(DEVICE).eval()

img_a_t = np_to_lpips_tensor(img_a, DEVICE)
img_b_t = np_to_lpips_tensor(img_b, DEVICE)
gt_t    = np_to_lpips_tensor(gt, DEVICE)

with torch.no_grad():
    lpips_a = lpips_fn(img_a_t, gt_t).item()
    lpips_b = lpips_fn(img_b_t, gt_t).item()


# ---------- 输出结果 ----------
print("========== Evaluation Results ==========")
print(f"A ({IMG_A_PATH})")
print(f"  PSNR : {psnr_a:.2f}")
print(f"  SSIM : {ssim_a:.4f}")
print(f"  LPIPS: {lpips_a:.4f}")
print()
print(f"B ({IMG_B_PATH})")
print(f"  PSNR : {psnr_b:.2f}")
print(f"  SSIM : {ssim_b:.4f}")
print(f"  LPIPS: {lpips_b:.4f}")
print("========================================")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /home/jian/miniconda3/envs/3dgs/lib/python3.9/site-packages/lpips/weights/v0.1/alex.pth
========== Evaluation Results ==========
A (fcgb_pro.png)
  PSNR : 18.78
  SSIM : 0.6751
  LPIPS: 0.2784

B (single_base.png)
  PSNR : 18.52
  SSIM : 0.6638
  LPIPS: 0.2814
